# SciRecQA — Notebook 1b: Negative-Class Construction & Detection Eval (standalone)

Runs entirely off the saved `scireqa_full_v1.1.jsonl` (upload the `llama-3.3-70b-instruct`
folder as a Kaggle dataset). **No OpenRouter / no Llama — free.** GPU + Internet ON (MiniCheck).

Pipeline (validated):
1. Load clean instances (rationales + their existing flan-t5 full-evidence confidences).
2. Load MiniCheck (`flan-t5-large`).
3. Build a **gold negative class** by controlled perturbation of the saved rationales —
   `fabricated_citation` (primary), `entity_swap`, and a `number_error` that **never touches
   `[...]` citation markers** (the bug from earlier runs). Score with **full multi-abstract
   evidence** (the config that gives fabricated AUC ≈ 0.98).
4. Eyeball the injected hallucinations.
5. Report detection **ROC-AUC** (overall + per type), pick a single **Youden** operating
   threshold, guard on **AUC ≥ 0.70**, relabel, and save.

Expected on this data: fabricated_citation AUC ≈ 0.98, sane threshold T ≈ 0.47, guard PASSES.

In [1]:
# == Wipe ALL previous Kaggle output (start fresh) ============================
# Removes everything under /kaggle/working: rationale checkpoints, saved JSONL
# benchmark files, stats, etc. /kaggle/input is read-only and is NOT touched.
import shutil
from pathlib import Path

WORKING = Path("/kaggle/working")

removed = []
for item in WORKING.iterdir():
    # keep nothing — clear checkpoints, benchmark files, caches, the lot
    try:
        if item.is_dir() and not item.is_symlink():
            shutil.rmtree(item, ignore_errors=True)
        else:
            item.unlink(missing_ok=True)
        removed.append(item.name)
    except Exception as e:
        print(f"  could not remove {item.name}: {e}")

print(f"Cleared /kaggle/working — removed {len(removed)} item(s):")
for name in sorted(removed):
    print("  -", name)
print("Remaining:", [p.name for p in WORKING.iterdir()] or "(empty)")


Cleared /kaggle/working — removed 9 item(s):
  - .virtual_documents
  - scireqa_full_v1.1.FIXED2_youden.jsonl
  - scireqa_full_v1.1.FIXED_defaults.jsonl
  - scireqa_full_with_negatives_v1.1.FIXED2_youden.jsonl
  - scireqa_full_with_negatives_v1.1.FIXED_defaults.jsonl
  - scireqa_hallucination_eval_v1.1.FIXED2_youden.jsonl
  - scireqa_hallucination_eval_v1.1.FIXED_defaults.jsonl
  - scireqa_hallucination_eval_v1.1.FIXED_v2.jsonl
  - state.db
Remaining: (empty)


In [2]:
# == 1. Imports + load clean instances ========================================
import json, re, os, sys, random
from pathlib import Path
from collections import Counter
import numpy as np

def _find(name):
    for root in ["/kaggle/input", "."]:
        r = Path(root)
        if r.exists():
            hits = list(r.rglob(name))
            if hits: return hits[0]
    return None

FULL_PATH = _find("scireqa_full_v1.1.jsonl")
OUT_DIR   = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert FULL_PATH, "scireqa_full_v1.1.jsonl not found - attach the dataset."
print("clean file:", FULL_PATH, "| output:", OUT_DIR)

clean = [json.loads(l) for l in open(FULL_PATH, encoding="utf-8") if l.strip()]
print("clean instances:", len(clean))

def save_jsonl(insts, path):
    with open(path, "w", encoding="utf-8") as f:
        for i in insts: f.write(json.dumps(i, ensure_ascii=False) + "\n")
    print("saved", path)

clean file: /kaggle/input/datasets/joytuntonny/llama-3-3-70b-instruct/scireqa_full_v1.1.jsonl | output: /kaggle/working
clean instances: 10000


In [3]:
# == 2. Load MiniCheck (flan-t5-large) ========================================
import importlib.util as _ilu, contextlib as _ctx
MODEL_E = "flan-t5-large"          # strongest atomic signal here (validated AUC~0.98 for fabricated)
if _ilu.find_spec("minicheck") is None:
    import subprocess as _sp
    _sp.run([sys.executable, "-m", "pip", "install", "-q",
             "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main"], check=False)
from minicheck.minicheck import MiniCheck
scorer = MiniCheck(model_name=MODEL_E, enable_prefix_caching=False)
with open(os.devnull, "w") as _dn, _ctx.redirect_stderr(_dn), _ctx.redirect_stdout(_dn):
    scorer.score(docs=["The sky is blue."], claims=["The sky is blue."])
print("MiniCheck loaded:", MODEL_E)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

MiniCheck loaded: flan-t5-large


In [4]:
# == 3. Build gold negative class + score (full-evidence) =====================
random.seed(42)
HALLUC_FRACTION = 0.30

_CITE = re.compile(r"\[[^\]]*\]")
def _spans(s):  return [m.span() for m in _CITE.finditer(s)]
def _inb(p,sp): return any(a <= p < b for a,b in sp)
def _sents(t):  return [s.strip() for s in re.split(r"(?<=[.!?])\s+", t.strip()) if len(s.strip()) >= 20]
def _tk(t):     return set(re.findall(r"[a-z]{3,}", (t or "").lower()))

_pool = list({(p.get("title") or "").strip()
              for i in clean for p in i.get("ground_truth_papers", [])
              if len((p.get("title") or "").strip()) > 12})

def full_ev(gt, max_chars=4000):
    parts = []
    for p in gt[:5]:
        t = (p.get("title") or "").strip(); ab = (p.get("abstract") or "").strip()
        if ab:  parts.append(f"{t}: {ab[:400]}")
        elif t: parts.append(t)
    return " ".join(parts)[:max_chars]

def p_number(s):
    sp = _spans(s)
    c = [m for m in re.finditer(r"\b(19|20)\d{2}\b", s) if not _inb(m.start(), sp)]
    if not c:
        c = [m for m in re.finditer(r"\b\d+(?:\.\d+)?%", s) if not _inb(m.start(), sp)]
    if not c: return s, False
    m = random.choice(c); g = m.group()
    new = (f"{round(float(g[:-1]) * random.choice([0.4,1.8,2.5]), 1)}%" if g.endswith("%")
           else str(int(g) + random.choice([-4,-3,3,4])))
    return s[:m.start()] + new + s[m.end():], True

def p_entity(s, gt):
    gt_t = {(p.get("title") or "").strip() for p in gt}
    q = [x for x in re.findall(r'"([^"]{8,})"', s) if any(x[:30] in t or t[:30] in x for t in gt_t)]
    pool = [t for t in _pool if t not in gt_t]
    if not q or not pool: return s, False
    return s.replace('"' + q[0] + '"', '"' + random.choice(pool) + '"', 1), True

def p_fab(gt):
    gt_t = {(p.get("title") or "").strip() for p in gt}
    pool = [t for t in _pool if t and t not in gt_t]
    if not pool: return None
    t = random.choice(pool); pct = random.choice([12,18,23,31,40])
    return random.choice([
        f'In particular, "{t}" reports a {pct}% improvement that these recommendations directly extend.',
        f'This line of work originates from "{t}", which the recommended papers cite as their main baseline.'])

def build_variant(inst):
    rat = inst.get("reference_rationale", "")
    if not rat or rat.startswith("["): return None
    pert = _sents(rat)
    if not pert: return None
    gi = pt = None
    for ty in random.sample(["entity_swap", "number_error", "fabricated_citation"], 3):
        if ty in ("entity_swap", "number_error"):
            order = list(range(len(pert))); random.shuffle(order)
            for i in order:
                ns, ok = (p_entity(pert[i], inst["ground_truth_papers"]) if ty == "entity_swap"
                          else p_number(pert[i]))
                if ok: pert[i] = ns; gi = i; pt = ty; break
        else:
            fab = p_fab(inst["ground_truth_papers"])
            if fab: pert.append(fab); gi = len(pert) - 1; pt = ty
        if gi is not None: break
    if gi is None: return None
    import copy
    v = copy.deepcopy(inst)
    v.update({"id": inst["id"] + "_h", "reference_rationale": " ".join(pert),
              "synthetic_hallucination": True, "perturbation_types": [pt],
              "source_instance_id": inst["id"], "annotation_status": "synthetic_gold",
              "_claims": pert, "_gi": gi})
    return v

cand = [i for i in clean if i.get("reference_rationale") and not i["reference_rationale"].startswith("[")]
random.shuffle(cand)
sel = cand[:int(len(cand) * HALLUC_FRACTION)]
variants = [v for v in (build_variant(i) for i in sel) if v]
print(f"Built {len(variants)} variants; types: {dict(Counter(v['perturbation_types'][0] for v in variants))}")

# score every claim against its instance's FULL multi-abstract evidence
pairs, docs, clm = [], [], []
ev_cache = {}
for vi, v in enumerate(variants):
    ev = ev_cache.setdefault(id(v), full_ev(v["ground_truth_papers"]))
    for ci, c in enumerate(v["_claims"]):
        if ev.strip():
            pairs.append((vi, ci)); docs.append(ev); clm.append(c)
conf, B = {}, 64
for s in range(0, len(pairs), B):
    with open(os.devnull, "w") as _dn, _ctx.redirect_stderr(_dn), _ctx.redirect_stdout(_dn):
        _, probs, _, _ = scorer.score(docs=docs[s:s+B], claims=clm[s:s+B])
    for (vi, ci), pr in zip(pairs[s:s+B], probs): conf[(vi, ci)] = float(pr)
    print(f"  scored {min(s+B, len(pairs))}/{len(pairs)}", end="\r")

for vi, v in enumerate(variants):
    labels = []
    for ci, c in enumerate(v["_claims"]):
        p = conf.get((vi, ci)); gold = "unsupported" if ci == v["_gi"] else "supported"
        labels.append({"claim_id": f"{ci:03d}", "claim_text": c, "claim_type": "factual_candidate",
                       "verdict": "pending" if p is None else ("supported" if p >= 0.5 else ("unsupported" if p < 0.2 else "neutral")),
                       "confidence": 0.0 if p is None else round(p, 4),
                       "label_type": "pending" if p is None else "auto_minicheck",
                       "model": f"MiniCheck-{MODEL_E}", "gold_verdict": gold,
                       "label_source": "synthetic_perturbation_v2",
                       "human_verified": False, "human_label": None})
    v["hallucination_labels"] = labels
    v.pop("_claims", None); v.pop("_gi", None)
neg = variants
save_jsonl(neg, OUT_DIR / "scireqa_hallucination_eval_v1.1.jsonl")

Built 3000 variants; types: {'fabricated_citation': 2335, 'entity_swap': 607, 'number_error': 58}
saved /kaggle/working/scireqa_hallucination_eval_v1.1.jsonl


In [5]:
# == 4. Eyeball the injected hallucinations ===================================
for k, v in enumerate(random.sample(neg, min(10, len(neg))), 1):
    gt = [(p.get("title") or "")[:60] for p in v.get("ground_truth_papers", [])[:3]]
    print(f"\n[{k}] type={v['perturbation_types']}  id={v['id']}")
    print(f"    GT: {gt}")
    for l in v["hallucination_labels"]:
        if l["gold_verdict"] == "unsupported":
            print(f"    >>> INJECTED (conf={l['confidence']}, verdict={l['verdict']}):")
            print(f"        {l['claim_text'][:160]}")


[1] type=['fabricated_citation']  id=e28634c17a63_h
    GT: ['4Seasons: Benchmarking Visual SLAM and Long-Term Localizatio', 'SPADE: Semi-supervised Anomaly Detection under Distribution ', 'A Modular Approach to the Embodiment of Hand Motions from Hu']
    >>> INJECTED (conf=0.4689, verdict=neutral):
        This line of work originates from "SPARK: Self-supervised Personalized Real-time Monocular Face Capture", which the recommended papers cite as their main baseli

[2] type=['entity_swap']  id=bc6e72ee0b43_h
    GT: ['HMD^2: Environment-aware Motion Generation from Single Egoce', 'Blendify -- Python rendering framework for Blender', 'SCRREAM : SCan, Register, REnder And Map:A Framework for Ann']
    >>> INJECTED (conf=0.4702, verdict=neutral):
        For instance, papers like "From Understanding to Utilization: A Survey on Explainability for Large Language Models" and "EgoCast: Forecasting Egocentric Human P

[3] type=['fabricated_citation']  id=3d3bd5924e27_h
    GT: ['Query-orien

In [6]:
# == 5. Detection AUC + Youden threshold + guard + save =======================
from sklearn.metrics import roc_auc_score

def pools(ptype=None):
    sup = [l["confidence"] for v in neg for l in v["hallucination_labels"]
           if l["gold_verdict"] == "supported" and l["label_type"] == "auto_minicheck"]
    uns = [l["confidence"] for v in neg for l in v["hallucination_labels"]
           if l["gold_verdict"] == "unsupported" and l["label_type"] == "auto_minicheck"
           and (ptype is None or v["perturbation_types"][0] == ptype)]
    return sup, uns

print("Detection AUC by perturbation type:")
for pt in ["fabricated_citation", "entity_swap", "number_error", None]:
    sup, uns = pools(pt)
    if sup and uns:
        a = roc_auc_score([0]*len(sup)+[1]*len(uns), [1-c for c in sup]+[1-c for c in uns])
        print(f"  {(pt or 'ALL'):20s}: AUC={a:.3f}  (n_unsup={len(uns)}, n_sup={len(sup)})")

sup, uns = pools(None)
auc = roc_auc_score([0]*len(sup)+[1]*len(uns), [1-c for c in sup]+[1-c for c in uns]) if (sup and uns) else 0.0
sa, ua = np.array(sup), np.array(uns)
T, bestJ = 0.5, None
for cnd in np.unique(np.concatenate([sa, ua])):
    J = float((ua < cnd).mean()) - float((sa < cnd).mean())
    if bestJ is None or J > bestJ: bestJ, T = J, float(cnd)
tpr = float((ua < T).mean()); fpr = float((sa < T).mean())
tp = int((ua < T).sum()); fp = int((sa < T).sum())
prec = tp / max(1, tp+fp); f1 = 2*prec*tpr / max(1e-9, prec+tpr)
print(f"\nOverall AUC = {auc:.3f}")
print(f"Youden operating point: flag UNSUPPORTED if confidence < {T:.4f}")
print(f"  recall(unsup)={tpr:.3f}  precision={prec:.3f}  F1={f1:.3f}  balanced-acc={0.5*(tpr+1-fpr):.3f}")

applied = auc >= 0.70
print("GUARD PASSED -> applying threshold." if applied
      else "GUARD FAILED (AUC<0.70) -> rely on human gold (nb2).")

# binary relabel from raw confidence (clean reuses its flan-t5 full-evidence confidence -> consistent)
def binarize(insts):
    for inst in insts:
        for l in inst.get("hallucination_labels", []):
            if str(l.get("label_type", "")).startswith("auto_minicheck"):
                l["verdict"] = "unsupported" if float(l.get("confidence", 1.0)) < T else "supported"
binarize(neg); binarize(clean)

def dist(insts, tag):
    L = [l for i in insts for l in i.get("hallucination_labels", [])]
    c = Counter(l["verdict"] for l in L); inf = c["supported"] + c["unsupported"]
    print(f"{tag}: claims={len(L)} " + " ".join(f"{k}={c[k]}({100*c[k]/max(1,len(L)):.1f}%)"
          for k in ["supported","unsupported","neutral","general_statement"]) +
          f" | pct_informative={100*inf/max(1,len(L)):.1f}%")
dist(clean, "CLEAN    "); dist(neg, "NEGATIVE ")

save_jsonl(clean,       OUT_DIR / "scireqa_full_v1.1.FINAL.jsonl")
save_jsonl(neg,         OUT_DIR / "scireqa_hallucination_eval_v1.1.FINAL.jsonl")
save_jsonl(clean + neg, OUT_DIR / "scireqa_full_with_negatives_v1.1.FINAL.jsonl")
print(f"\nHEADLINE detection metric: AUC = {auc:.3f} (operating threshold T={T:.4f}).")
print("Report AUC as the headline; T is the operating point. Download the *.FINAL.jsonl files.")

Detection AUC by perturbation type:
  fabricated_citation : AUC=0.974  (n_unsup=2335, n_sup=11108)
  entity_swap         : AUC=0.874  (n_unsup=607, n_sup=11108)
  number_error        : AUC=0.748  (n_unsup=58, n_sup=11108)
  ALL                 : AUC=0.949  (n_unsup=3000, n_sup=11108)

Overall AUC = 0.949
Youden operating point: flag UNSUPPORTED if confidence < 0.4720
  recall(unsup)=0.912  precision=0.617  F1=0.736  balanced-acc=0.880
GUARD PASSED -> applying threshold.
CLEAN    : claims=39287 supported=24424(62.2%) unsupported=4481(11.4%) neutral=0(0.0%) general_statement=10382(26.4%) | pct_informative=73.6%
NEGATIVE : claims=14108 supported=9674(68.6%) unsupported=4434(31.4%) neutral=0(0.0%) general_statement=0(0.0%) | pct_informative=100.0%
saved /kaggle/working/scireqa_full_v1.1.FINAL.jsonl
saved /kaggle/working/scireqa_hallucination_eval_v1.1.FINAL.jsonl
saved /kaggle/working/scireqa_full_with_negatives_v1.1.FINAL.jsonl

HEADLINE detection metric: AUC = 0.949 (operating threshold 

## 6. Control — is the fabricated_citation signal grounding or a template artifact?

`fabricated_citation` negatives all use a fixed template, so high AUC could mean MiniCheck
reacts to the *phrasing*, not the missing grounding. Minimal-pair test: keep the exact
template but swap the unrelated title for a **real GT title** (grounded entity) and re-score.
If that raises the score toward the natural-supported level, detection is grounding-based and
the AUC is genuine; if not, it's a style artifact and you should headline `entity_swap` (0.874).

In [ ]:
# == 6. Same-template control: grounding vs template artifact =================
import os, contextlib, random, re
import numpy as np
random.seed(1)

fab = [v for v in neg if v["perturbation_types"] == ["fabricated_citation"]]
fake_conf, ctrl_docs, ctrl_clm = [], [], []
for v in fab:
    inj = next((l for l in v["hallucination_labels"] if l["gold_verdict"] == "unsupported"), None)
    m   = re.search(r'"([^"]+)"', inj["claim_text"]) if inj else None
    gts = [p.get("title", "") for p in v["ground_truth_papers"] if p.get("title")]
    if not (inj and m and gts):
        continue
    fake_conf.append(inj["confidence"])
    real = random.choice(gts)                                       # a REAL GT title (grounded)
    ctrl_clm.append(inj["claim_text"].replace('"' + m.group(1) + '"', '"' + real + '"', 1))  # same template
    ctrl_docs.append(full_ev(v["ground_truth_papers"]))

real_conf = []
for s in range(0, len(ctrl_clm), 64):
    with open(os.devnull, "w") as _dn, contextlib.redirect_stderr(_dn), contextlib.redirect_stdout(_dn):
        _, probs, _, _ = scorer.score(docs=ctrl_docs[s:s+64], claims=ctrl_clm[s:s+64])
    real_conf.extend(float(p) for p in probs)
fake_conf = fake_conf[:len(real_conf)]

nat_sup = [l["confidence"] for v in neg for l in v["hallucination_labels"]
           if l["gold_verdict"] == "supported" and l["label_type"] == "auto_minicheck"]
mf, mr, mn = np.mean(fake_conf), np.mean(real_conf), np.mean(nat_sup)
gap_closed = (mr - mf) / max(1e-9, (mn - mf))
print(f"fabricated (fake title)     mean conf = {mf:.4f}  (n={len(fake_conf)})")
print(f"CONTROL    (real GT title)  mean conf = {mr:.4f}")
print(f"natural supported           mean conf = {mn:.4f}")
print(f"\nfraction of the fake->supported gap closed by using a real grounded title: {gap_closed:.2f}")
if gap_closed > 0.5:
    print("=> Real grounded title RAISES the score substantially -> detection is GROUNDING-based.")
    print("   The fabricated AUC (and the ~0.95 overall) is genuine, not a template artifact.")
else:
    print("=> Real title barely changes the score -> MiniCheck reacts to TEMPLATE STYLE.")
    print("   Artifact confirmed. Headline entity_swap (0.874); treat fabricated as secondary.")